# Vignette-level performance

Describe accuracy and confidence before and after resource consultation.

## Pre-consultation performance

### Summarize pre-consultation condition accuracy

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("..").resolve() / "data" / "Psy_Case_Vignette_Data_Prolific_analysis.xlsx"
VIGNETTES = ["A", "B", "C", "D", "E"]

PRE_CONDITION_COLS = [f"vig_{v}_pre_condition_correct" for v in VIGNETTES]

df = pd.read_excel(DATA_PATH)

missing_cols = [col for col in PRE_CONDITION_COLS if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}. Available columns: {list(df.columns)}")

rows = []
for col in PRE_CONDITION_COLS:
    series = df[col]
    n_people = len(series)
    n_correct = (series == 1).sum()
    pct_correct = n_correct / n_people * 100 if n_people else float("nan")

    vignette = col.split("_")[1]
    rows.append(
        {
            "vignette": vignette,
            "column": col,
            "n_people": n_people,
            "n_correct": n_correct,
            "pct_correct": round(pct_correct, 2),
        }
    )

pre_condition_summary = pd.DataFrame(rows)

pooled = df[PRE_CONDITION_COLS]
n_responses = pooled.notna().sum().sum()
n_correct_total = (pooled == 1).sum().sum()
pct_correct_total = n_correct_total / n_responses * 100 if n_responses else float("nan")

print("By vignette:")
print(pre_condition_summary.to_string(index=False))
print()
print("All together (pooled across all 5 pre-condition items):")
print(f"  Correct responses: {n_correct_total:,} of {n_responses:,} ({pct_correct_total:.2f}%)")

### Model pre-consultation condition accuracy

In [ ]:
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

N_ITEMS = len(PRE_CONDITION_COLS)

df["total_pre_condition_correct"] = df[PRE_CONDITION_COLS].sum(axis=1)
df["prop_pre_condition_correct"] = df["total_pre_condition_correct"] / N_ITEMS

reg_df = df[["prop_pre_condition_correct", "total_pre_condition_correct", "Age", "Sex"]].dropna()

print(f"Outcome: total_pre_condition_correct (sum of {N_ITEMS} pre-condition items)")
print(reg_df["total_pre_condition_correct"].describe().round(2))
print()

model = smf.glm(
    "prop_pre_condition_correct ~ Age + C(Sex, Treatment(reference='Female'))",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
result = model.fit()
print(result.summary())

### Compare pre-condition accuracy by country

In [ ]:
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

COUNTRY_COL = "Country of residence"
N_ITEMS = len(PRE_CONDITION_COLS)

if COUNTRY_COL not in df.columns:
    raise KeyError(f"Column not found: {COUNTRY_COL!r}. Available columns: {list(df.columns)}")

if "total_pre_condition_correct" not in df.columns:
    df["total_pre_condition_correct"] = df[PRE_CONDITION_COLS].sum(axis=1)

work = df[[COUNTRY_COL, "total_pre_condition_correct", *PRE_CONDITION_COLS]].dropna(
    subset=[COUNTRY_COL, "total_pre_condition_correct"]
)
countries = work[COUNTRY_COL].value_counts().index.tolist()

print("Pre-condition diagnosis by country of residence")
print("=" * 60)

# --- Sample level: each participant's total correct (0–5) ---
sample_summary = (
    work.groupby(COUNTRY_COL, observed=True)["total_pre_condition_correct"]
    .agg(n="count", mean="mean", sd="std", median="median", min="min", max="max")
    .round(2)
)
print("\nSample level — total correct per participant (0–5):")
print(sample_summary.to_string())

if len(countries) == 2:
    g1, g2 = countries
    s1 = work.loc[work[COUNTRY_COL] == g1, "total_pre_condition_correct"]
    s2 = work.loc[work[COUNTRY_COL] == g2, "total_pre_condition_correct"]
    mw = stats.mannwhitneyu(s1, s2, alternative="two-sided")
    print(
        f"\nMann–Whitney U ({g1} vs {g2}): "
        f"U = {mw.statistic:.1f}, p = {mw.pvalue:.4f}"
    )

reg_df = work[[COUNTRY_COL, "total_pre_condition_correct"]].copy()
reg_df["prop_pre_condition_correct"] = reg_df["total_pre_condition_correct"] / N_ITEMS
reg_df = reg_df.rename(columns={COUNTRY_COL: "country"})
glm = smf.glm(
    "prop_pre_condition_correct ~ C(country)",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
glm_result = glm.fit()
print("\nBinomial GLM (total correct out of 5 ~ country):")
print(glm_result.summary().tables[1])

# --- Total level: pooled correct/incorrect across all pre-condition items ---
print("\n" + "=" * 60)
print("Total level — pooled across all pre-condition items:")

rows = []
for country in countries:
    sub = work.loc[work[COUNTRY_COL] == country, PRE_CONDITION_COLS]
    n_responses = sub.notna().sum().sum()
    n_correct = (sub == 1).sum().sum()
    pct_correct = n_correct / n_responses * 100 if n_responses else float("nan")
    rows.append(
        {
            "country": country,
            "n_participants": len(sub),
            "n_responses": int(n_responses),
            "n_correct": int(n_correct),
            "pct_correct": round(pct_correct, 2),
        }
    )

pooled_summary = pd.DataFrame(rows)
print(pooled_summary.to_string(index=False))

if len(countries) == 2:
    table = np.zeros((2, 2), dtype=int)
    for i, country in enumerate(countries):
        sub = work.loc[work[COUNTRY_COL] == country, PRE_CONDITION_COLS]
        table[i, 0] = (sub == 1).sum().sum()
        table[i, 1] = (sub == 0).sum().sum()
    chi2, p_chi, _, _ = stats.chi2_contingency(table)
    print(
        f"\nChi-square on pooled item responses ({countries[0]} vs {countries[1]}): "
        f"chi2 = {chi2:.2f}, p = {p_chi:.4f}"
    )

print("\nBy vignette (% correct by country):")
vignette_rows = []
for col in PRE_CONDITION_COLS:
    vignette = col.split("_")[1]
    row = {"vignette": vignette}
    for country in countries:
        sub = work.loc[work[COUNTRY_COL] == country, col]
        n = sub.notna().sum()
        pct = (sub == 1).sum() / n * 100 if n else float("nan")
        row[f"{country} %"] = round(pct, 2)
    if len(countries) == 2:
        sub_v = work[[COUNTRY_COL, col]].dropna()
        ct = pd.crosstab(sub_v[COUNTRY_COL], sub_v[col])
        if ct.shape == (2, 2):
            chi2_v, p_v, _, _ = stats.chi2_contingency(ct.values)
            row["chi2 p"] = round(p_v, 4)
    vignette_rows.append(row)

print(pd.DataFrame(vignette_rows).to_string(index=False))

### Compare pre-condition accuracy by study arm

In [ ]:
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

ARM_LABELS = {
    1: "ChatGPT",
    2: "Copilot",
    3: "Perplexity",
    4: "Control",
}
ARM_ORDER = ["ChatGPT", "Copilot", "Perplexity", "Control"]
ARM_REF = 4
N_ITEMS = len(PRE_CONDITION_COLS)

if "study_arm" not in df.columns:
    raise KeyError(f"Column not found: 'study_arm'. Available columns: {list(df.columns)}")

if "total_pre_condition_correct" not in df.columns:
    df["total_pre_condition_correct"] = df[PRE_CONDITION_COLS].sum(axis=1)

work = df[["study_arm", "total_pre_condition_correct", *PRE_CONDITION_COLS]].dropna(
    subset=["study_arm", "total_pre_condition_correct"]
)
work = work.copy()
work["Arm"] = pd.Categorical(
    work["study_arm"].map(ARM_LABELS),
    categories=ARM_ORDER,
    ordered=True,
)

print("Pre-condition diagnosis by study arm")
print("=" * 60)

# --- Sample level: each participant's total correct (0–5) ---
sample_summary = (
    work.groupby("Arm", observed=True)["total_pre_condition_correct"]
    .agg(n="count", mean="mean", sd="std", median="median", min="min", max="max")
    .round(2)
)
print("\nSample level — total correct per participant (0–5):")
print(sample_summary.to_string())

arm_groups = [
    group["total_pre_condition_correct"].to_numpy()
    for _, group in work.groupby("Arm", observed=True)
]
if all(len(g) > 0 for g in arm_groups):
    h_stat, p_kw = stats.kruskal(*arm_groups)
    print(f"\nKruskal–Wallis test across arms: H = {h_stat:.2f}, p = {p_kw:.4f}")

reg_df = work[["study_arm", "total_pre_condition_correct"]].copy()
reg_df["prop_pre_condition_correct"] = reg_df["total_pre_condition_correct"] / N_ITEMS
reg_df["study_arm"] = reg_df["study_arm"].astype(int)
glm = smf.glm(
    f"prop_pre_condition_correct ~ C(study_arm, Treatment(reference={ARM_REF}))",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
glm_result = glm.fit()
print("\nBinomial GLM (total correct out of 5 ~ study arm; reference = Control):")
print(glm_result.summary().tables[1])

# --- Total level: pooled correct/incorrect across all pre-condition items ---
print("\n" + "=" * 60)
print("Total level — pooled across all pre-condition items:")

rows = []
for arm_name in ARM_ORDER:
    sub = work.loc[work["Arm"] == arm_name, PRE_CONDITION_COLS]
    n_responses = sub.notna().sum().sum()
    n_correct = (sub == 1).sum().sum()
    pct_correct = n_correct / n_responses * 100 if n_responses else float("nan")
    rows.append(
        {
            "arm": arm_name,
            "n_participants": len(sub),
            "n_responses": int(n_responses),
            "n_correct": int(n_correct),
            "pct_correct": round(pct_correct, 2),
        }
    )

pooled_summary = pd.DataFrame(rows)
print(pooled_summary.to_string(index=False))

table = np.zeros((len(ARM_ORDER), 2), dtype=int)
for i, arm_name in enumerate(ARM_ORDER):
    sub = work.loc[work["Arm"] == arm_name, PRE_CONDITION_COLS]
    table[i, 0] = (sub == 1).sum().sum()
    table[i, 1] = (sub == 0).sum().sum()
chi2, p_chi, _, _ = stats.chi2_contingency(table)
print(f"\nChi-square on pooled item responses across arms: chi2 = {chi2:.2f}, p = {p_chi:.4f}")

print("\nBy vignette (% correct by study arm):")
vignette_rows = []
for col in PRE_CONDITION_COLS:
    vignette = col.split("_")[1]
    row = {"vignette": vignette}
    for arm_name in ARM_ORDER:
        sub = work.loc[work["Arm"] == arm_name, col]
        n = sub.notna().sum()
        pct = (sub == 1).sum() / n * 100 if n else float("nan")
        row[f"{arm_name} %"] = round(pct, 2)
    sub_v = work[["Arm", col]].dropna()
    ct = pd.crosstab(sub_v["Arm"], sub_v[col])
    ct = ct.reindex(index=ARM_ORDER, fill_value=0)
    if ct.shape[0] >= 2 and ct.shape[1] >= 2:
        chi2_v, p_v, _, _ = stats.chi2_contingency(ct.values)
        row["chi2 p"] = round(p_v, 4)
    vignette_rows.append(row)

print(pd.DataFrame(vignette_rows).to_string(index=False))

### Summarize pre-consultation triage accuracy

In [ ]:
PRE_NEXTSTEP_COLS = [f"vig_{v}_pre_nextstep_correct" for v in VIGNETTES]

missing_cols = [col for col in PRE_NEXTSTEP_COLS if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}. Available columns: {list(df.columns)}")

rows = []
for col in PRE_NEXTSTEP_COLS:
    series = df[col]
    n_people = len(series)
    n_correct = (series == 1).sum()
    pct_correct = n_correct / n_people * 100 if n_people else float("nan")

    vignette = col.split("_")[1]
    rows.append(
        {
            "vignette": vignette,
            "column": col,
            "n_people": n_people,
            "n_correct": n_correct,
            "pct_correct": round(pct_correct, 2),
        }
    )

pre_nextstep_summary = pd.DataFrame(rows)

pooled = df[PRE_NEXTSTEP_COLS]
n_responses = pooled.notna().sum().sum()
n_correct_total = (pooled == 1).sum().sum()
pct_correct_total = n_correct_total / n_responses * 100 if n_responses else float("nan")

print("By vignette:")
print(pre_nextstep_summary.to_string(index=False))
print()
print("All together (pooled across all 5 pre-nextstep items):")
print(f"  Correct responses: {n_correct_total:,} of {n_responses:,} ({pct_correct_total:.2f}%)")

### Model pre-consultation triage accuracy

In [ ]:
N_ITEMS = len(PRE_NEXTSTEP_COLS)

df["total_pre_nextstep_correct"] = df[PRE_NEXTSTEP_COLS].sum(axis=1)
df["prop_pre_nextstep_correct"] = df["total_pre_nextstep_correct"] / N_ITEMS

reg_df = df[["prop_pre_nextstep_correct", "total_pre_nextstep_correct", "Age", "Sex"]].dropna()

print(f"Outcome: total_pre_nextstep_correct (sum of {N_ITEMS} pre-nextstep items)")
print(reg_df["total_pre_nextstep_correct"].describe().round(2))
print()

model = smf.glm(
    "prop_pre_nextstep_correct ~ Age + C(Sex, Treatment(reference='Female'))",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
result = model.fit()
print(result.summary())

### Compare pre-triage accuracy by country

In [ ]:
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

COUNTRY_COL = "Country of residence"
N_ITEMS = len(PRE_NEXTSTEP_COLS)

if COUNTRY_COL not in df.columns:
    raise KeyError(f"Column not found: {COUNTRY_COL!r}. Available columns: {list(df.columns)}")

if "total_pre_nextstep_correct" not in df.columns:
    df["total_pre_nextstep_correct"] = df[PRE_NEXTSTEP_COLS].sum(axis=1)

work = df[[COUNTRY_COL, "total_pre_nextstep_correct", *PRE_NEXTSTEP_COLS]].dropna(
    subset=[COUNTRY_COL, "total_pre_nextstep_correct"]
)
countries = work[COUNTRY_COL].value_counts().index.tolist()

print("Pre-nextstep (triage) by country of residence")
print("=" * 60)

# --- Sample level: each participant's total correct (0–5) ---
sample_summary = (
    work.groupby(COUNTRY_COL, observed=True)["total_pre_nextstep_correct"]
    .agg(n="count", mean="mean", sd="std", median="median", min="min", max="max")
    .round(2)
)
print("\nSample level — total correct per participant (0–5):")
print(sample_summary.to_string())

if len(countries) == 2:
    g1, g2 = countries
    s1 = work.loc[work[COUNTRY_COL] == g1, "total_pre_nextstep_correct"]
    s2 = work.loc[work[COUNTRY_COL] == g2, "total_pre_nextstep_correct"]
    mw = stats.mannwhitneyu(s1, s2, alternative="two-sided")
    print(
        f"\nMann–Whitney U ({g1} vs {g2}): "
        f"U = {mw.statistic:.1f}, p = {mw.pvalue:.4f}"
    )

reg_df = work[[COUNTRY_COL, "total_pre_nextstep_correct"]].copy()
reg_df["prop_pre_nextstep_correct"] = reg_df["total_pre_nextstep_correct"] / N_ITEMS
reg_df = reg_df.rename(columns={COUNTRY_COL: "country"})
glm = smf.glm(
    "prop_pre_nextstep_correct ~ C(country)",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
glm_result = glm.fit()
print("\nBinomial GLM (total correct out of 5 ~ country):")
print(glm_result.summary().tables[1])

# --- Total level: pooled correct/incorrect across all pre-nextstep items ---
print("\n" + "=" * 60)
print("Total level — pooled across all pre-nextstep items:")

rows = []
for country in countries:
    sub = work.loc[work[COUNTRY_COL] == country, PRE_NEXTSTEP_COLS]
    n_responses = sub.notna().sum().sum()
    n_correct = (sub == 1).sum().sum()
    pct_correct = n_correct / n_responses * 100 if n_responses else float("nan")
    rows.append(
        {
            "country": country,
            "n_participants": len(sub),
            "n_responses": int(n_responses),
            "n_correct": int(n_correct),
            "pct_correct": round(pct_correct, 2),
        }
    )

pooled_summary = pd.DataFrame(rows)
print(pooled_summary.to_string(index=False))

if len(countries) == 2:
    table = np.zeros((2, 2), dtype=int)
    for i, country in enumerate(countries):
        sub = work.loc[work[COUNTRY_COL] == country, PRE_NEXTSTEP_COLS]
        table[i, 0] = (sub == 1).sum().sum()
        table[i, 1] = (sub == 0).sum().sum()
    chi2, p_chi, _, _ = stats.chi2_contingency(table)
    print(
        f"\nChi-square on pooled item responses ({countries[0]} vs {countries[1]}): "
        f"chi2 = {chi2:.2f}, p = {p_chi:.4f}"
    )

print("\nBy vignette (% correct by country):")
vignette_rows = []
for col in PRE_NEXTSTEP_COLS:
    vignette = col.split("_")[1]
    row = {"vignette": vignette}
    for country in countries:
        sub = work.loc[work[COUNTRY_COL] == country, col]
        n = sub.notna().sum()
        pct = (sub == 1).sum() / n * 100 if n else float("nan")
        row[f"{country} %"] = round(pct, 2)
    if len(countries) == 2:
        sub_v = work[[COUNTRY_COL, col]].dropna()
        ct = pd.crosstab(sub_v[COUNTRY_COL], sub_v[col])
        if ct.shape == (2, 2):
            chi2_v, p_v, _, _ = stats.chi2_contingency(ct.values)
            row["chi2 p"] = round(p_v, 4)
    vignette_rows.append(row)

print(pd.DataFrame(vignette_rows).to_string(index=False))

### Compare pre-triage accuracy by study arm

In [ ]:
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

ARM_LABELS = {
    1: "ChatGPT",
    2: "Copilot",
    3: "Perplexity",
    4: "Control",
}
ARM_ORDER = ["ChatGPT", "Copilot", "Perplexity", "Control"]
ARM_REF = 4
N_ITEMS = len(PRE_NEXTSTEP_COLS)

if "study_arm" not in df.columns:
    raise KeyError(f"Column not found: 'study_arm'. Available columns: {list(df.columns)}")

if "total_pre_nextstep_correct" not in df.columns:
    df["total_pre_nextstep_correct"] = df[PRE_NEXTSTEP_COLS].sum(axis=1)

work = df[["study_arm", "total_pre_nextstep_correct", *PRE_NEXTSTEP_COLS]].dropna(
    subset=["study_arm", "total_pre_nextstep_correct"]
)
work = work.copy()
work["Arm"] = pd.Categorical(
    work["study_arm"].map(ARM_LABELS),
    categories=ARM_ORDER,
    ordered=True,
)

print("Pre-nextstep (triage) by study arm")
print("=" * 60)

# --- Sample level: each participant's total correct (0–5) ---
sample_summary = (
    work.groupby("Arm", observed=True)["total_pre_nextstep_correct"]
    .agg(n="count", mean="mean", sd="std", median="median", min="min", max="max")
    .round(2)
)
print("\nSample level — total correct per participant (0–5):")
print(sample_summary.to_string())

arm_groups = [
    group["total_pre_nextstep_correct"].to_numpy()
    for _, group in work.groupby("Arm", observed=True)
]
if all(len(g) > 0 for g in arm_groups):
    h_stat, p_kw = stats.kruskal(*arm_groups)
    print(f"\nKruskal–Wallis test across arms: H = {h_stat:.2f}, p = {p_kw:.4f}")

reg_df = work[["study_arm", "total_pre_nextstep_correct"]].copy()
reg_df["prop_pre_nextstep_correct"] = reg_df["total_pre_nextstep_correct"] / N_ITEMS
reg_df["study_arm"] = reg_df["study_arm"].astype(int)
glm = smf.glm(
    f"prop_pre_nextstep_correct ~ C(study_arm, Treatment(reference={ARM_REF}))",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
glm_result = glm.fit()
print("\nBinomial GLM (total correct out of 5 ~ study arm; reference = Control):")
print(glm_result.summary().tables[1])

# --- Total level: pooled correct/incorrect across all pre-nextstep items ---
print("\n" + "=" * 60)
print("Total level — pooled across all pre-nextstep items:")

rows = []
for arm_name in ARM_ORDER:
    sub = work.loc[work["Arm"] == arm_name, PRE_NEXTSTEP_COLS]
    n_responses = sub.notna().sum().sum()
    n_correct = (sub == 1).sum().sum()
    pct_correct = n_correct / n_responses * 100 if n_responses else float("nan")
    rows.append(
        {
            "arm": arm_name,
            "n_participants": len(sub),
            "n_responses": int(n_responses),
            "n_correct": int(n_correct),
            "pct_correct": round(pct_correct, 2),
        }
    )

pooled_summary = pd.DataFrame(rows)
print(pooled_summary.to_string(index=False))

table = np.zeros((len(ARM_ORDER), 2), dtype=int)
for i, arm_name in enumerate(ARM_ORDER):
    sub = work.loc[work["Arm"] == arm_name, PRE_NEXTSTEP_COLS]
    table[i, 0] = (sub == 1).sum().sum()
    table[i, 1] = (sub == 0).sum().sum()
chi2, p_chi, _, _ = stats.chi2_contingency(table)
print(f"\nChi-square on pooled item responses across arms: chi2 = {chi2:.2f}, p = {p_chi:.4f}")

print("\nBy vignette (% correct by study arm):")
vignette_rows = []
for col in PRE_NEXTSTEP_COLS:
    vignette = col.split("_")[1]
    row = {"vignette": vignette}
    for arm_name in ARM_ORDER:
        sub = work.loc[work["Arm"] == arm_name, col]
        n = sub.notna().sum()
        pct = (sub == 1).sum() / n * 100 if n else float("nan")
        row[f"{arm_name} %"] = round(pct, 2)
    sub_v = work[["Arm", col]].dropna()
    ct = pd.crosstab(sub_v["Arm"], sub_v[col])
    ct = ct.reindex(index=ARM_ORDER, fill_value=0)
    if ct.shape[0] >= 2 and ct.shape[1] >= 2:
        chi2_v, p_v, _, _ = stats.chi2_contingency(ct.values)
        row["chi2 p"] = round(p_v, 4)
    vignette_rows.append(row)

print(pd.DataFrame(vignette_rows).to_string(index=False))

### Summarize pre-consultation confidence

In [ ]:
PRE_CONFIDENCE_COLS = [f"vig_{v}_pre_confidence" for v in VIGNETTES]

missing_cols = [col for col in PRE_CONFIDENCE_COLS if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}. Available columns: {list(df.columns)}")

rows = []
for col in PRE_CONFIDENCE_COLS:
    series = df[col]
    rows.append(
        {
            "vignette": col.split("_")[1],
            "column": col,
            "n": series.notna().sum(),
            "mean": round(series.mean(), 2),
            "std": round(series.std(), 2),
            "median": round(series.median(), 2),
            "min": series.min(),
            "max": series.max(),
        }
    )

pre_confidence_summary = pd.DataFrame(rows)
df["mean_pre_confidence"] = df[PRE_CONFIDENCE_COLS].mean(axis=1)

pooled = df[PRE_CONFIDENCE_COLS]
n_responses = pooled.notna().sum().sum()

print("By vignette:")
print(pre_confidence_summary.to_string(index=False))
print()
print("Overall (pooled across all 5 pre-confidence ratings):")
print(f"  Mean: {pooled.stack().mean():.2f}")
print(f"  Std:  {pooled.stack().std():.2f}")
print(f"  N:    {n_responses:,}")
print()
print("Overall (mean pre-confidence per participant, averaged across vignettes A–E):")
print(df["mean_pre_confidence"].describe().round(2))

### Model pre-condition accuracy with confidence

In [ ]:
N_ITEMS = len(PRE_CONDITION_COLS)

reg_df = df[
    ["prop_pre_condition_correct", "total_pre_condition_correct", "Age", "Sex", "mean_pre_confidence"]
].dropna()

print(f"Outcome: total_pre_condition_correct (sum of {N_ITEMS} pre-condition items)")
print(f"Predictors: Age, Sex, mean pre-confidence")
print()

model = smf.glm(
    "prop_pre_condition_correct ~ Age + C(Sex, Treatment(reference='Female')) + mean_pre_confidence",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
result = model.fit()
print(result.summary())

### Model pre-triage accuracy with confidence

In [ ]:
N_ITEMS = len(PRE_NEXTSTEP_COLS)

reg_df = df[
    ["prop_pre_nextstep_correct", "total_pre_nextstep_correct", "Age", "Sex", "mean_pre_confidence"]
].dropna()

print(f"Outcome: total_pre_nextstep_correct (sum of {N_ITEMS} pre-nextstep items)")
print(f"Predictors: Age, Sex, mean pre-confidence")
print()

model = smf.glm(
    "prop_pre_nextstep_correct ~ Age + C(Sex, Treatment(reference='Female')) + mean_pre_confidence",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
result = model.fit()
print(result.summary())

### Model pre-consultation confidence

In [ ]:
reg_df = df[["mean_pre_confidence", "Age", "Sex"]].dropna()

print("Outcome: mean_pre_confidence (average of 5 pre-confidence ratings, 0–100)")
print(reg_df["mean_pre_confidence"].describe().round(2))
print()

model = smf.ols(
    "mean_pre_confidence ~ Age + C(Sex, Treatment(reference='Female'))",
    data=reg_df,
)
result = model.fit()
print(result.summary())

## Summarize post-consultation condition accuracy

In [ ]:
POST_CONDITION_COLS = [f"vig_{v}_post_condition_correct" for v in VIGNETTES]

missing_cols = [col for col in POST_CONDITION_COLS if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}. Available columns: {list(df.columns)}")

rows = []
for col in POST_CONDITION_COLS:
    series = df[col]
    n_people = len(series)
    n_correct = (series == 1).sum()
    pct_correct = n_correct / n_people * 100 if n_people else float("nan")

    vignette = col.split("_")[1]
    rows.append(
        {
            "vignette": vignette,
            "column": col,
            "n_people": n_people,
            "n_correct": n_correct,
            "pct_correct": round(pct_correct, 2),
        }
    )

post_condition_summary = pd.DataFrame(rows)

pooled = df[POST_CONDITION_COLS]
n_responses = pooled.notna().sum().sum()
n_correct_total = (pooled == 1).sum().sum()
pct_correct_total = n_correct_total / n_responses * 100 if n_responses else float("nan")

print("By vignette:")
print(post_condition_summary.to_string(index=False))
print()
print("All together (pooled across all 5 post-condition items):")
print(f"  Correct responses: {n_correct_total:,} of {n_responses:,} ({pct_correct_total:.2f}%)")

### Model post-consultation condition accuracy

In [ ]:
N_ITEMS = len(POST_CONDITION_COLS)

df["total_post_condition_correct"] = df[POST_CONDITION_COLS].sum(axis=1)
df["prop_post_condition_correct"] = df["total_post_condition_correct"] / N_ITEMS

reg_df = df[["prop_post_condition_correct", "total_post_condition_correct", "Age", "Sex"]].dropna()

print(f"Outcome: total_post_condition_correct (sum of {N_ITEMS} post-condition items)")
print(reg_df["total_post_condition_correct"].describe().round(2))
print()

model = smf.glm(
    "prop_post_condition_correct ~ Age + C(Sex, Treatment(reference='Female'))",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
result = model.fit()
print(result.summary())

### Compare post-condition accuracy by country

In [ ]:
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

COUNTRY_COL = "Country of residence"
N_ITEMS = len(POST_CONDITION_COLS)

if COUNTRY_COL not in df.columns:
    raise KeyError(f"Column not found: {COUNTRY_COL!r}. Available columns: {list(df.columns)}")

if "total_post_condition_correct" not in df.columns:
    df["total_post_condition_correct"] = df[POST_CONDITION_COLS].sum(axis=1)

work = df[[COUNTRY_COL, "total_post_condition_correct", *POST_CONDITION_COLS]].dropna(
    subset=[COUNTRY_COL, "total_post_condition_correct"]
)
countries = work[COUNTRY_COL].value_counts().index.tolist()

print("Post-condition diagnosis by country of residence")
print("=" * 60)

# --- Sample level: each participant's total correct (0–5) ---
sample_summary = (
    work.groupby(COUNTRY_COL, observed=True)["total_post_condition_correct"]
    .agg(n="count", mean="mean", sd="std", median="median", min="min", max="max")
    .round(2)
)
print("\nSample level — total correct per participant (0–5):")
print(sample_summary.to_string())

if len(countries) == 2:
    g1, g2 = countries
    s1 = work.loc[work[COUNTRY_COL] == g1, "total_post_condition_correct"]
    s2 = work.loc[work[COUNTRY_COL] == g2, "total_post_condition_correct"]
    mw = stats.mannwhitneyu(s1, s2, alternative="two-sided")
    print(
        f"\nMann–Whitney U ({g1} vs {g2}): "
        f"U = {mw.statistic:.1f}, p = {mw.pvalue:.4f}"
    )

reg_df = work[[COUNTRY_COL, "total_post_condition_correct"]].copy()
reg_df["prop_post_condition_correct"] = reg_df["total_post_condition_correct"] / N_ITEMS
reg_df = reg_df.rename(columns={COUNTRY_COL: "country"})
glm = smf.glm(
    "prop_post_condition_correct ~ C(country)",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
glm_result = glm.fit()
print("\nBinomial GLM (total correct out of 5 ~ country):")
print(glm_result.summary().tables[1])

# --- Total level: pooled correct/incorrect across all post-condition items ---
print("\n" + "=" * 60)
print("Total level — pooled across all post-condition items:")

rows = []
for country in countries:
    sub = work.loc[work[COUNTRY_COL] == country, POST_CONDITION_COLS]
    n_responses = sub.notna().sum().sum()
    n_correct = (sub == 1).sum().sum()
    pct_correct = n_correct / n_responses * 100 if n_responses else float("nan")
    rows.append(
        {
            "country": country,
            "n_participants": len(sub),
            "n_responses": int(n_responses),
            "n_correct": int(n_correct),
            "pct_correct": round(pct_correct, 2),
        }
    )

pooled_summary = pd.DataFrame(rows)
print(pooled_summary.to_string(index=False))

if len(countries) == 2:
    table = np.zeros((2, 2), dtype=int)
    for i, country in enumerate(countries):
        sub = work.loc[work[COUNTRY_COL] == country, POST_CONDITION_COLS]
        table[i, 0] = (sub == 1).sum().sum()
        table[i, 1] = (sub == 0).sum().sum()
    chi2, p_chi, _, _ = stats.chi2_contingency(table)
    print(
        f"\nChi-square on pooled item responses ({countries[0]} vs {countries[1]}): "
        f"chi2 = {chi2:.2f}, p = {p_chi:.4f}"
    )

print("\nBy vignette (% correct by country):")
vignette_rows = []
for col in POST_CONDITION_COLS:
    vignette = col.split("_")[1]
    row = {"vignette": vignette}
    for country in countries:
        sub = work.loc[work[COUNTRY_COL] == country, col]
        n = sub.notna().sum()
        pct = (sub == 1).sum() / n * 100 if n else float("nan")
        row[f"{country} %"] = round(pct, 2)
    if len(countries) == 2:
        sub_v = work[[COUNTRY_COL, col]].dropna()
        ct = pd.crosstab(sub_v[COUNTRY_COL], sub_v[col])
        if ct.shape == (2, 2):
            chi2_v, p_v, _, _ = stats.chi2_contingency(ct.values)
            row["chi2 p"] = round(p_v, 4)
    vignette_rows.append(row)

print(pd.DataFrame(vignette_rows).to_string(index=False))

### Compare pre- and post-consultation condition accuracy

In [ ]:
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

if "total_pre_condition_correct" not in df.columns:
    df["total_pre_condition_correct"] = df[PRE_CONDITION_COLS].sum(axis=1)
if "total_post_condition_correct" not in df.columns:
    df["total_post_condition_correct"] = df[POST_CONDITION_COLS].sum(axis=1)

rows = []
for v in VIGNETTES:
    pre = df[f"vig_{v}_pre_condition_correct"]
    post = df[f"vig_{v}_post_condition_correct"]

    n_pre_correct = (pre == 1).sum()
    n_post_correct = (post == 1).sum()
    n_people = len(df)
    pre_pct = n_pre_correct / n_people * 100
    post_pct = n_post_correct / n_people * 100

    pre_wrong_post_right = ((pre == 0) & (post == 1)).sum()
    pre_right_post_wrong = ((pre == 1) & (post == 0)).sum()
    table = [
        [((pre == 0) & (post == 0)).sum(), pre_wrong_post_right],
        [pre_right_post_wrong, ((pre == 1) & (post == 1)).sum()],
    ]
    mcnemar_result = mcnemar(table, exact=False, correction=True)

    rows.append(
        {
            "vignette": v,
            "pre_pct_correct": round(pre_pct, 2),
            "post_pct_correct": round(post_pct, 2),
            "pct_change": round(post_pct - pre_pct, 2),
            "pre_wrong_post_right": pre_wrong_post_right,
            "pre_right_post_wrong": pre_right_post_wrong,
            "mcnemar_p": round(mcnemar_result.pvalue, 4),
        }
    )

vignette_comparison = pd.DataFrame(rows)

print("By vignette (McNemar's test):")
print(vignette_comparison.to_string(index=False))
print()

pre_pooled = df[PRE_CONDITION_COLS]
post_pooled = df[POST_CONDITION_COLS]
pre_pooled_pct = (pre_pooled == 1).sum().sum() / pre_pooled.notna().sum().sum() * 100
post_pooled_pct = (post_pooled == 1).sum().sum() / post_pooled.notna().sum().sum() * 100

pre_total = df["total_pre_condition_correct"]
post_total = df["total_post_condition_correct"]
wilcoxon_result = stats.wilcoxon(pre_total, post_total, zero_method="wilcox", correction=True)

print("Overall:")
print(f"  Pooled pre accuracy:  {pre_pooled_pct:.2f}%")
print(f"  Pooled post accuracy: {post_pooled_pct:.2f}%")
print(f"  Pooled change:        {post_pooled_pct - pre_pooled_pct:+.2f} pp")
print()
print(f"  Mean total pre correct (0–5):  {pre_total.mean():.2f}")
print(f"  Mean total post correct (0–5): {post_total.mean():.2f}")
print(f"  Mean change:                   {(post_total - pre_total).mean():+.2f}")
print()
print(f"  Wilcoxon signed-rank test (paired totals): statistic={wilcoxon_result.statistic:.1f}, p={wilcoxon_result.pvalue:.4f}")

## Summarize post-consultation triage accuracy

In [ ]:
POST_NEXTSTEP_COLS = [f"vig_{v}_post_nextstep_correct" for v in VIGNETTES]

missing_cols = [col for col in POST_NEXTSTEP_COLS if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}. Available columns: {list(df.columns)}")

rows = []
for col in POST_NEXTSTEP_COLS:
    series = df[col]
    n_people = len(series)
    n_correct = (series == 1).sum()
    pct_correct = n_correct / n_people * 100 if n_people else float("nan")

    vignette = col.split("_")[1]
    rows.append(
        {
            "vignette": vignette,
            "column": col,
            "n_people": n_people,
            "n_correct": n_correct,
            "pct_correct": round(pct_correct, 2),
        }
    )

post_nextstep_summary = pd.DataFrame(rows)

pooled = df[POST_NEXTSTEP_COLS]
n_responses = pooled.notna().sum().sum()
n_correct_total = (pooled == 1).sum().sum()
pct_correct_total = n_correct_total / n_responses * 100 if n_responses else float("nan")

print("By vignette:")
print(post_nextstep_summary.to_string(index=False))
print()
print("All together (pooled across all 5 post-nextstep items):")
print(f"  Correct responses: {n_correct_total:,} of {n_responses:,} ({pct_correct_total:.2f}%)")

### Model post-consultation triage accuracy

In [ ]:
N_ITEMS = len(POST_NEXTSTEP_COLS)

df["total_post_nextstep_correct"] = df[POST_NEXTSTEP_COLS].sum(axis=1)
df["prop_post_nextstep_correct"] = df["total_post_nextstep_correct"] / N_ITEMS

reg_df = df[["prop_post_nextstep_correct", "total_post_nextstep_correct", "Age", "Sex"]].dropna()

print(f"Outcome: total_post_nextstep_correct (sum of {N_ITEMS} post-nextstep items)")
print(reg_df["total_post_nextstep_correct"].describe().round(2))
print()

model = smf.glm(
    "prop_post_nextstep_correct ~ Age + C(Sex, Treatment(reference='Female'))",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
result = model.fit()
print(result.summary())

### Compare post-triage accuracy by country

In [ ]:
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

COUNTRY_COL = "Country of residence"
N_ITEMS = len(POST_NEXTSTEP_COLS)

if COUNTRY_COL not in df.columns:
    raise KeyError(f"Column not found: {COUNTRY_COL!r}. Available columns: {list(df.columns)}")

if "total_post_nextstep_correct" not in df.columns:
    df["total_post_nextstep_correct"] = df[POST_NEXTSTEP_COLS].sum(axis=1)

work = df[[COUNTRY_COL, "total_post_nextstep_correct", *POST_NEXTSTEP_COLS]].dropna(
    subset=[COUNTRY_COL, "total_post_nextstep_correct"]
)
countries = work[COUNTRY_COL].value_counts().index.tolist()

print("Post-nextstep (triage) by country of residence")
print("=" * 60)

# --- Sample level: each participant's total correct (0–5) ---
sample_summary = (
    work.groupby(COUNTRY_COL, observed=True)["total_post_nextstep_correct"]
    .agg(n="count", mean="mean", sd="std", median="median", min="min", max="max")
    .round(2)
)
print("\nSample level — total correct per participant (0–5):")
print(sample_summary.to_string())

if len(countries) == 2:
    g1, g2 = countries
    s1 = work.loc[work[COUNTRY_COL] == g1, "total_post_nextstep_correct"]
    s2 = work.loc[work[COUNTRY_COL] == g2, "total_post_nextstep_correct"]
    mw = stats.mannwhitneyu(s1, s2, alternative="two-sided")
    print(
        f"\nMann–Whitney U ({g1} vs {g2}): "
        f"U = {mw.statistic:.1f}, p = {mw.pvalue:.4f}"
    )

reg_df = work[[COUNTRY_COL, "total_post_nextstep_correct"]].copy()
reg_df["prop_post_nextstep_correct"] = reg_df["total_post_nextstep_correct"] / N_ITEMS
reg_df = reg_df.rename(columns={COUNTRY_COL: "country"})
glm = smf.glm(
    "prop_post_nextstep_correct ~ C(country)",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
glm_result = glm.fit()
print("\nBinomial GLM (total correct out of 5 ~ country):")
print(glm_result.summary().tables[1])

# --- Total level: pooled correct/incorrect across all post-nextstep items ---
print("\n" + "=" * 60)
print("Total level — pooled across all post-nextstep items:")

rows = []
for country in countries:
    sub = work.loc[work[COUNTRY_COL] == country, POST_NEXTSTEP_COLS]
    n_responses = sub.notna().sum().sum()
    n_correct = (sub == 1).sum().sum()
    pct_correct = n_correct / n_responses * 100 if n_responses else float("nan")
    rows.append(
        {
            "country": country,
            "n_participants": len(sub),
            "n_responses": int(n_responses),
            "n_correct": int(n_correct),
            "pct_correct": round(pct_correct, 2),
        }
    )

pooled_summary = pd.DataFrame(rows)
print(pooled_summary.to_string(index=False))

if len(countries) == 2:
    table = np.zeros((2, 2), dtype=int)
    for i, country in enumerate(countries):
        sub = work.loc[work[COUNTRY_COL] == country, POST_NEXTSTEP_COLS]
        table[i, 0] = (sub == 1).sum().sum()
        table[i, 1] = (sub == 0).sum().sum()
    chi2, p_chi, _, _ = stats.chi2_contingency(table)
    print(
        f"\nChi-square on pooled item responses ({countries[0]} vs {countries[1]}): "
        f"chi2 = {chi2:.2f}, p = {p_chi:.4f}"
    )

print("\nBy vignette (% correct by country):")
vignette_rows = []
for col in POST_NEXTSTEP_COLS:
    vignette = col.split("_")[1]
    row = {"vignette": vignette}
    for country in countries:
        sub = work.loc[work[COUNTRY_COL] == country, col]
        n = sub.notna().sum()
        pct = (sub == 1).sum() / n * 100 if n else float("nan")
        row[f"{country} %"] = round(pct, 2)
    if len(countries) == 2:
        sub_v = work[[COUNTRY_COL, col]].dropna()
        ct = pd.crosstab(sub_v[COUNTRY_COL], sub_v[col])
        if ct.shape == (2, 2):
            chi2_v, p_v, _, _ = stats.chi2_contingency(ct.values)
            row["chi2 p"] = round(p_v, 4)
    vignette_rows.append(row)

print(pd.DataFrame(vignette_rows).to_string(index=False))

### Compare pre- and post-consultation triage accuracy

In [ ]:
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

if "total_pre_nextstep_correct" not in df.columns:
    df["total_pre_nextstep_correct"] = df[PRE_NEXTSTEP_COLS].sum(axis=1)
if "total_post_nextstep_correct" not in df.columns:
    df["total_post_nextstep_correct"] = df[POST_NEXTSTEP_COLS].sum(axis=1)

rows = []
for v in VIGNETTES:
    pre = df[f"vig_{v}_pre_nextstep_correct"]
    post = df[f"vig_{v}_post_nextstep_correct"]

    n_pre_correct = (pre == 1).sum()
    n_post_correct = (post == 1).sum()
    n_people = len(df)
    pre_pct = n_pre_correct / n_people * 100
    post_pct = n_post_correct / n_people * 100

    pre_wrong_post_right = ((pre == 0) & (post == 1)).sum()
    pre_right_post_wrong = ((pre == 1) & (post == 0)).sum()
    table = [
        [((pre == 0) & (post == 0)).sum(), pre_wrong_post_right],
        [pre_right_post_wrong, ((pre == 1) & (post == 1)).sum()],
    ]
    mcnemar_result = mcnemar(table, exact=False, correction=True)

    rows.append(
        {
            "vignette": v,
            "pre_pct_correct": round(pre_pct, 2),
            "post_pct_correct": round(post_pct, 2),
            "pct_change": round(post_pct - pre_pct, 2),
            "pre_wrong_post_right": pre_wrong_post_right,
            "pre_right_post_wrong": pre_right_post_wrong,
            "mcnemar_p": round(mcnemar_result.pvalue, 4),
        }
    )

nextstep_comparison = pd.DataFrame(rows)

print("By vignette (McNemar's test):")
print(nextstep_comparison.to_string(index=False))
print()

pre_pooled = df[PRE_NEXTSTEP_COLS]
post_pooled = df[POST_NEXTSTEP_COLS]
pre_pooled_pct = (pre_pooled == 1).sum().sum() / pre_pooled.notna().sum().sum() * 100
post_pooled_pct = (post_pooled == 1).sum().sum() / post_pooled.notna().sum().sum() * 100

pre_total = df["total_pre_nextstep_correct"]
post_total = df["total_post_nextstep_correct"]
wilcoxon_result = stats.wilcoxon(pre_total, post_total, zero_method="wilcox", correction=True)

print("Overall:")
print(f"  Pooled pre accuracy:  {pre_pooled_pct:.2f}%")
print(f"  Pooled post accuracy: {post_pooled_pct:.2f}%")
print(f"  Pooled change:        {post_pooled_pct - pre_pooled_pct:+.2f} pp")
print()
print(f"  Mean total pre correct (0–5):  {pre_total.mean():.2f}")
print(f"  Mean total post correct (0–5): {post_total.mean():.2f}")
print(f"  Mean change:                   {(post_total - pre_total).mean():+.2f}")
print()
print(f"  Wilcoxon signed-rank test (paired totals): statistic={wilcoxon_result.statistic:.1f}, p={wilcoxon_result.pvalue:.4f}")

## Summarize post-consultation confidence

In [ ]:
POST_CONFIDENCE_COLS = [f"vig_{v}_post_confidence" for v in VIGNETTES]

missing_cols = [col for col in POST_CONFIDENCE_COLS if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}. Available columns: {list(df.columns)}")

rows = []
for col in POST_CONFIDENCE_COLS:
    series = df[col]
    rows.append(
        {
            "vignette": col.split("_")[1],
            "column": col,
            "n": series.notna().sum(),
            "mean": round(series.mean(), 2),
            "std": round(series.std(), 2),
            "median": round(series.median(), 2),
            "min": series.min(),
            "max": series.max(),
        }
    )

post_confidence_summary = pd.DataFrame(rows)
df["mean_post_confidence"] = df[POST_CONFIDENCE_COLS].mean(axis=1)

pooled = df[POST_CONFIDENCE_COLS]
n_responses = pooled.notna().sum().sum()

print("By vignette:")
print(post_confidence_summary.to_string(index=False))
print()
print("Overall (pooled across all 5 post-confidence ratings):")
print(f"  Mean: {pooled.stack().mean():.2f}")
print(f"  Std:  {pooled.stack().std():.2f}")
print(f"  N:    {n_responses:,}")
print()
print("Overall (mean post-confidence per participant, averaged across vignettes A–E):")
print(df["mean_post_confidence"].describe().round(2))

### Model post-condition accuracy with confidence

In [ ]:
N_ITEMS = len(POST_CONDITION_COLS)

if "total_post_condition_correct" not in df.columns:
    df["total_post_condition_correct"] = df[POST_CONDITION_COLS].sum(axis=1)
    df["prop_post_condition_correct"] = df["total_post_condition_correct"] / N_ITEMS

reg_df = df[
    ["prop_post_condition_correct", "total_post_condition_correct", "Age", "Sex", "mean_post_confidence"]
].dropna()

print(f"Outcome: total_post_condition_correct (sum of {N_ITEMS} post-condition items)")
print(f"Predictors: Age, Sex, mean post-confidence")
print()

model = smf.glm(
    "prop_post_condition_correct ~ Age + C(Sex, Treatment(reference='Female')) + mean_post_confidence",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
result = model.fit()
print(result.summary())

### Model post-triage accuracy with confidence

In [ ]:
N_ITEMS = len(POST_NEXTSTEP_COLS)

if "total_post_nextstep_correct" not in df.columns:
    df["total_post_nextstep_correct"] = df[POST_NEXTSTEP_COLS].sum(axis=1)
    df["prop_post_nextstep_correct"] = df["total_post_nextstep_correct"] / N_ITEMS

reg_df = df[
    ["prop_post_nextstep_correct", "total_post_nextstep_correct", "Age", "Sex", "mean_post_confidence"]
].dropna()

print(f"Outcome: total_post_nextstep_correct (sum of {N_ITEMS} post-nextstep items)")
print(f"Predictors: Age, Sex, mean post-confidence")
print()

model = smf.glm(
    "prop_post_nextstep_correct ~ Age + C(Sex, Treatment(reference='Female')) + mean_post_confidence",
    data=reg_df,
    family=sm.families.Binomial(),
    var_weights=np.full(len(reg_df), N_ITEMS),
)
result = model.fit()
print(result.summary())

### Model post-consultation confidence

In [ ]:
reg_df = df[["mean_post_confidence", "Age", "Sex"]].dropna()

print("Outcome: mean_post_confidence (average of 5 post-confidence ratings, 0–100)")
print(reg_df["mean_post_confidence"].describe().round(2))
print()

model = smf.ols(
    "mean_post_confidence ~ Age + C(Sex, Treatment(reference='Female'))",
    data=reg_df,
)
result = model.fit()
print(result.summary())

### Compare pre- and post-consultation confidence

In [ ]:
from scipy import stats

if "mean_pre_confidence" not in df.columns:
    df["mean_pre_confidence"] = df[PRE_CONFIDENCE_COLS].mean(axis=1)
if "mean_post_confidence" not in df.columns:
    df["mean_post_confidence"] = df[POST_CONFIDENCE_COLS].mean(axis=1)

rows = []
for v in VIGNETTES:
    pre = df[f"vig_{v}_pre_confidence"]
    post = df[f"vig_{v}_post_confidence"]

    pre_mean = pre.mean()
    post_mean = post.mean()
    mean_change = (post - pre).mean()
    wilcoxon_result = stats.wilcoxon(pre, post, zero_method="wilcox", correction=True)

    rows.append(
        {
            "vignette": v,
            "pre_mean": round(pre_mean, 2),
            "post_mean": round(post_mean, 2),
            "mean_change": round(mean_change, 2),
            "wilcoxon_p": round(wilcoxon_result.pvalue, 4),
        }
    )

confidence_comparison = pd.DataFrame(rows)

print("By vignette (Wilcoxon signed-rank test):")
print(confidence_comparison.to_string(index=False))
print()

pre_pooled = df[PRE_CONFIDENCE_COLS]
post_pooled = df[POST_CONFIDENCE_COLS]
pre_pooled_mean = pre_pooled.stack().mean()
post_pooled_mean = post_pooled.stack().mean()

pre_mean = df["mean_pre_confidence"]
post_mean = df["mean_post_confidence"]
wilcoxon_result = stats.wilcoxon(pre_mean, post_mean, zero_method="wilcox", correction=True)

print("Overall:")
print(f"  Pooled pre mean:  {pre_pooled_mean:.2f}")
print(f"  Pooled post mean: {post_pooled_mean:.2f}")
print(f"  Pooled change:    {post_pooled_mean - pre_pooled_mean:+.2f}")
print()
print(f"  Mean pre-confidence per participant:  {pre_mean.mean():.2f}")
print(f"  Mean post-confidence per participant: {post_mean.mean():.2f}")
print(f"  Mean change:                          {(post_mean - pre_mean).mean():+.2f}")
print()
print(f"  Wilcoxon signed-rank test (paired participant means): statistic={wilcoxon_result.statistic:.1f}, p={wilcoxon_result.pvalue:.4f}")

## Sensitivity analyses

### Compare performance change by country

In [ ]:
import numpy as np
from scipy import stats

COUNTRY_COL = "Country of residence"

if "total_pre_condition_correct" not in df.columns:
    df["total_pre_condition_correct"] = df[PRE_CONDITION_COLS].sum(axis=1)
if "total_post_condition_correct" not in df.columns:
    df["total_post_condition_correct"] = df[POST_CONDITION_COLS].sum(axis=1)
if "total_pre_nextstep_correct" not in df.columns:
    df["total_pre_nextstep_correct"] = df[PRE_NEXTSTEP_COLS].sum(axis=1)
if "total_post_nextstep_correct" not in df.columns:
    df["total_post_nextstep_correct"] = df[POST_NEXTSTEP_COLS].sum(axis=1)

if COUNTRY_COL not in df.columns:
    raise KeyError(f"Column not found: {COUNTRY_COL!r}. Available columns: {list(df.columns)}")


def summarize_pre_post_change(data, group_col, pre_col, post_col, task_label):
    change_col = f"change_{pre_col}"
    work = data[[group_col, pre_col, post_col]].dropna().copy()
    work[change_col] = work[post_col] - work[pre_col]

    rows = []
    for group, group_df in work.groupby(group_col, sort=False):
        wilcoxon_result = stats.wilcoxon(
            group_df[pre_col],
            group_df[post_col],
            zero_method="wilcox",
            correction=True,
        )
        rows.append(
            {
                "group": group,
                "n": len(group_df),
                "mean_pre": round(group_df[pre_col].mean(), 2),
                "mean_post": round(group_df[post_col].mean(), 2),
                "mean_change": round(group_df[change_col].mean(), 2),
                "within_group_wilcoxon_p": round(wilcoxon_result.pvalue, 4),
            }
        )

    summary = pd.DataFrame(rows)
    groups = summary["group"].tolist()

    print(f"{task_label}:")
    print(summary.to_string(index=False))

    if len(groups) == 2:
        g1, g2 = groups
        change1 = work.loc[work[group_col] == g1, change_col]
        change2 = work.loc[work[group_col] == g2, change_col]
        mw_result = stats.mannwhitneyu(change1, change2, alternative="two-sided")
        print(
            f"  Mann–Whitney U on change scores ({g1} vs {g2}): "
            f"U={mw_result.statistic:.1f}, p={mw_result.pvalue:.4f}"
        )
    print()


summarize_pre_post_change(
    df,
    COUNTRY_COL,
    "total_pre_condition_correct",
    "total_post_condition_correct",
    "Condition (pre vs post change by country)",
)
summarize_pre_post_change(
    df,
    COUNTRY_COL,
    "total_pre_nextstep_correct",
    "total_post_nextstep_correct",
    "Nextstep (pre vs post change by country)",
)

### Compare performance change by attention-check status

In [ ]:
ATT_COLS = ["att_check_fail", "att_check2_fail"]
missing_cols = [col for col in ATT_COLS if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}. Available columns: {list(df.columns)}")

df["att_check_failed"] = (df["att_check_fail"] == 1) | (df["att_check2_fail"] == 1)
df["att_check_group"] = np.where(df["att_check_failed"], "Failed at least once", "No failures")

print(f"Attention-check groups:")
print(df["att_check_group"].value_counts().to_string())
print()

summarize_pre_post_change(
    df,
    "att_check_group",
    "total_pre_condition_correct",
    "total_post_condition_correct",
    "Condition (pre vs post change by attention-check group)",
)
summarize_pre_post_change(
    df,
    "att_check_group",
    "total_pre_nextstep_correct",
    "total_post_nextstep_correct",
    "Nextstep (pre vs post change by attention-check group)",
)

## Plot pre- and post-consultation accuracy by vignette

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.8,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    }
)

DATA_PATH = Path("..").resolve() / "data" / "Psy_Case_Vignette_Data_Prolific_analysis.xlsx"
VIGNETTES = ["A", "B", "C", "D", "E"]
PRE_CONDITION_COLS = [f"vig_{v}_pre_condition_correct" for v in VIGNETTES]
POST_CONDITION_COLS = [f"vig_{v}_post_condition_correct" for v in VIGNETTES]
PRE_NEXTSTEP_COLS = [f"vig_{v}_pre_nextstep_correct" for v in VIGNETTES]
POST_NEXTSTEP_COLS = [f"vig_{v}_post_nextstep_correct" for v in VIGNETTES]

df = pd.read_excel(DATA_PATH)
df["total_pre_condition_correct"] = df[PRE_CONDITION_COLS].sum(axis=1)
df["total_post_condition_correct"] = df[POST_CONDITION_COLS].sum(axis=1)
df["total_pre_nextstep_correct"] = df[PRE_NEXTSTEP_COLS].sum(axis=1)
df["total_post_nextstep_correct"] = df[POST_NEXTSTEP_COLS].sum(axis=1)

FIG_DIR = Path("..").resolve() / "manuscript" / "Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
PNG_PATH = FIG_DIR / "Figure_2_pre_post_accuracy.png"
PDF_PATH = FIG_DIR / "Figure_2_pre_post_accuracy.pdf"

METRICS = [
    {
        "title": "Condition diagnosis",
        "pre_cols": PRE_CONDITION_COLS,
        "post_cols": POST_CONDITION_COLS,
        "pre_prefix": "pre_condition",
        "post_prefix": "post_condition",
        "total_pre_col": "total_pre_condition_correct",
        "total_post_col": "total_post_condition_correct",
    },
    {
        "title": "Triage (next step)",
        "pre_cols": PRE_NEXTSTEP_COLS,
        "post_cols": POST_NEXTSTEP_COLS,
        "pre_prefix": "pre_nextstep",
        "post_prefix": "post_nextstep",
        "total_pre_col": "total_pre_nextstep_correct",
        "total_post_col": "total_post_nextstep_correct",
    },
]

PRE_COLOR = "#636363"
POST_COLOR = "#2166ac"
LINE_COLOR = "#bdbdbd"
CI_COLOR = "#888888"


def wilson_ci_pct(successes: int, n: int, alpha: float = 0.05) -> tuple[float, float]:
    if n == 0:
        return float("nan"), float("nan")
    lo, hi = proportion_confint(successes, n, alpha=alpha, method="wilson")
    return lo * 100, hi * 100


def proportion_row(label: str, successes: int, n: int, mcnemar_p: float = np.nan) -> dict:
    pct = successes / n * 100 if n else float("nan")
    lo, hi = wilson_ci_pct(successes, n)
    return {
        "label": label,
        "n": n,
        "successes": successes,
        "pct": pct,
        "ci_lo": lo,
        "ci_hi": hi,
        "mcnemar_p": mcnemar_p,
    }


def mcnemar_p(pre: pd.Series, post: pd.Series) -> float:
    table = [
        [((pre == 0) & (post == 0)).sum(), ((pre == 0) & (post == 1)).sum()],
        [((pre == 1) & (post == 0)).sum(), ((pre == 1) & (post == 1)).sum()],
    ]
    return float(mcnemar(table, exact=False, correction=True).pvalue)


def build_metric_summary(metric: dict) -> pd.DataFrame:
    pre_pooled = df[metric["pre_cols"]]
    post_pooled = df[metric["post_cols"]]
    pre_n = int(pre_pooled.notna().sum().sum())
    post_n = int(post_pooled.notna().sum().sum())
    pre_success = int((pre_pooled == 1).sum().sum())
    post_success = int((post_pooled == 1).sum().sum())

    rows = [
        {
            **proportion_row("Overall", pre_success, pre_n),
            "phase": "pre",
        },
        {
            **proportion_row("Overall", post_success, post_n),
            "phase": "post",
        },
    ]

    vignette_rows = []
    for v in VIGNETTES:
        pre = df[f"vig_{v}_{metric['pre_prefix']}_correct"]
        post = df[f"vig_{v}_{metric['post_prefix']}_correct"]
        p = mcnemar_p(pre, post)
        pre_n_v = int(pre.notna().sum())
        post_n_v = int(post.notna().sum())
        vignette_rows.append({**proportion_row(v, int((pre == 1).sum()), pre_n_v, p), "phase": "pre"})
        vignette_rows.append({**proportion_row(v, int((post == 1).sum()), post_n_v, p), "phase": "post"})

    pre_total = df[metric["total_pre_col"]]
    post_total = df[metric["total_post_col"]]
    wilcoxon_p = stats.wilcoxon(pre_total, post_total, zero_method="wilcox", correction=True).pvalue

    summary = pd.DataFrame([rows[0], rows[1]] + vignette_rows)
    summary.loc[(summary["label"] == "Overall") & (summary["phase"] == "pre"), "wilcoxon_p"] = wilcoxon_p
    summary.loc[(summary["label"] == "Overall") & (summary["phase"] == "post"), "wilcoxon_p"] = wilcoxon_p
    return summary


def plot_table(summary: pd.DataFrame) -> pd.DataFrame:
    """Wide table for printing: one row per label with pre/post pct and CIs."""
    out = []
    for label in summary["label"].unique():
        sub = summary[summary["label"] == label]
        pre = sub[sub["phase"] == "pre"].iloc[0]
        post = sub[sub["phase"] == "post"].iloc[0]
        out.append(
            {
                "label": label,
                "pre_pct": pre["pct"],
                "pre_ci": f"[{pre['ci_lo']:.1f}, {pre['ci_hi']:.1f}]",
                "post_pct": post["pct"],
                "post_ci": f"[{post['ci_lo']:.1f}, {post['ci_hi']:.1f}]",
                "mcnemar_p": pre["mcnemar_p"],
                "wilcoxon_p": pre.get("wilcoxon_p", np.nan),
            }
        )
    return pd.DataFrame(out)


def p_to_stars(p: float) -> str:
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def x_positions(labels: list[str]) -> dict[str, float]:
    if not labels:
        return {}
    xs = [0.0] + [1.4 + i for i in range(len(labels) - 1)]
    return dict(zip(labels, xs))


def plot_dumbbell_panel(ax, summary: pd.DataFrame, panel_title: str, *, show_ylabel: bool) -> None:
    labels = summary["label"].unique().tolist()
    x_map = x_positions(labels)
    offset = 0.16
    max_top = 0.0

    for label in labels:
        sub = summary[summary["label"] == label]
        pre = sub[sub["phase"] == "pre"].iloc[0]
        post = sub[sub["phase"] == "post"].iloc[0]
        xi = x_map[label]

        ax.plot(
            [xi - offset, xi + offset],
            [pre["pct"], post["pct"]],
            color=LINE_COLOR,
            lw=1.5,
            solid_capstyle="round",
            zorder=1,
        )

        for x_pt, row, marker_cfg in [
            (xi - offset, pre, {"markerfacecolor": "white", "markeredgecolor": PRE_COLOR}),
            (xi + offset, post, {"markerfacecolor": POST_COLOR, "markeredgecolor": POST_COLOR}),
        ]:
            y = row["pct"]
            yerr = np.array([[y - row["ci_lo"]], [row["ci_hi"] - y]])
            ax.errorbar(
                x_pt,
                y,
                yerr=yerr,
                fmt="o",
                markersize=6.5,
                markeredgewidth=1.4,
                color=CI_COLOR,
                elinewidth=1.0,
                capsize=3,
                capthick=1.0,
                zorder=3,
                **marker_cfg,
            )
            max_top = max(max_top, row["ci_hi"])

        p_val = pre["mcnemar_p"] if label != "Overall" else pre.get("wilcoxon_p", np.nan)
        stars = p_to_stars(p_val)
        if stars:
            ax.text(xi, max(pre["ci_hi"], post["ci_hi"]) + 3.5, stars, ha="center", va="bottom", fontsize=9, color="#333333")
            max_top = max(max_top, max(pre["ci_hi"], post["ci_hi"]) + 3.5)

    xs = [x_map[l] for l in labels]
    if len(xs) > 1:
        ax.axvline((xs[0] + xs[1]) / 2, color="#e6e6e6", lw=1.0, linestyle="--", zorder=0)

    ax.set_title(panel_title, loc="left", fontsize=11, fontweight="bold", pad=8)
    ax.set_xticks(xs)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Percent correct" if show_ylabel else "")
    ax.set_ylim(0, min(100, max_top + 8))
    ax.set_xlim(xs[0] - 0.55, xs[-1] + 0.55)
    ax.yaxis.grid(True, linestyle="-", alpha=0.25, color="#cccccc")
    ax.set_axisbelow(True)


summaries = {}
for metric in METRICS:
    summary = build_metric_summary(metric)
    summaries[metric["title"]] = summary
    print(f"{metric['title']}:")
    print(plot_table(summary).round({"pre_pct": 2, "post_pct": 2, "mcnemar_p": 4, "wilcoxon_p": 4}).to_string(index=False))
    print()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8), sharey=True)
fig.subplots_adjust(wspace=0.08, top=0.88, bottom=0.14, left=0.08, right=0.98)

for ax, metric in zip(axes, METRICS):
    plot_dumbbell_panel(ax, summaries[metric["title"]], metric["title"], show_ylabel=(ax is axes[0]))

pre_handle = plt.Line2D(
    [0],
    [0],
    marker="o",
    color="w",
    markerfacecolor="white",
    markeredgecolor=PRE_COLOR,
    markeredgewidth=1.4,
    markersize=6.5,
    label="Pre",
    linestyle="None",
)
post_handle = plt.Line2D(
    [0],
    [0],
    marker="o",
    color="w",
    markerfacecolor=POST_COLOR,
    markeredgecolor=POST_COLOR,
    markersize=6.5,
    label="Post",
    linestyle="None",
)
fig.legend(
    handles=[pre_handle, post_handle],
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.0),
    handletextpad=0.4,
    columnspacing=1.5,
)

fig.savefig(PNG_PATH, dpi=300, bbox_inches="tight")
fig.savefig(PDF_PATH, bbox_inches="tight")
plt.show()

print(f"Saved {PNG_PATH}")
print(f"Saved {PDF_PATH}")



## Plot pre- and post-consultation accuracy by study arm

In [ ]:
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rpy2.robjects import default_converter, pandas2ri, r
from rpy2.robjects.conversion import localconverter
from statsmodels.stats.proportion import proportion_confint

DATA_PATH = Path("..").resolve() / "data" / "Psy_Case_Vignette_Data_Prolific_analysis.xlsx"
FIG_DIR = Path("..").resolve() / "manuscript" / "Figures"
PNG_PATH = FIG_DIR / "Figure_3_pre_post_by_arm.png"
PDF_PATH = FIG_DIR / "Figure_3_pre_post_by_arm.pdf"

VIGNETTES = ["A", "B", "C", "D", "E"]
ARM_ORDER = ["ChatGPT", "Copilot", "Perplexity", "Control"]
ARM_LABELS = {"1": "ChatGPT", "2": "Copilot", "3": "Perplexity", "4": "Control"}
ARM_COLORS = {
    "ChatGPT": "#009E73",
    "Copilot": "#0072B2",
    "Perplexity": "#E69F00",
    "Control": "#999999",
}
ARM_JITTER = {"ChatGPT": -0.075, "Copilot": -0.025, "Perplexity": 0.025, "Control": 0.075}
CI_COLOR = "#888888"
PHASE_X = {"Pre": 0.0, "Post": 1.0}
FIG3_OUTCOMES = [
    {
        "outcome": "Condition diagnosis",
        "pre_cols": [f"vig_{v}_pre_condition_correct" for v in VIGNETTES],
        "post_cols": [f"vig_{v}_post_condition_correct" for v in VIGNETTES],
    },
    {
        "outcome": "Triage (next step)",
        "pre_cols": [f"vig_{v}_pre_nextstep_correct" for v in VIGNETTES],
        "post_cols": [f"vig_{v}_post_nextstep_correct" for v in VIGNETTES],
    },
]


def wilson_ci_pct(successes: int, n: int, alpha: float = 0.05) -> tuple[float, float]:
    if n == 0:
        return float("nan"), float("nan")
    lo, hi = proportion_confint(successes, n, alpha=alpha, method="wilson")
    return lo * 100, hi * 100


def p_to_stars(p: float) -> str:
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def make_long_outcome(df: pd.DataFrame, pre_cols: list[str], post_cols: list[str]) -> pd.DataFrame:
    parts = []
    for v, pre_col, post_col in zip(VIGNETTES, pre_cols, post_cols):
        for post, col in ((0, pre_col), (1, post_col)):
            parts.append(
                df.assign(vignette=v, post=post, correct=df[col])[
                    ["prolific_id", "study_arm", "Age", "Sex", "vignette", "post", "correct"]
                ]
            )
    long_df = pd.concat(parts, ignore_index=True)
    return (
        long_df.dropna(subset=["correct", "study_arm", "prolific_id", "Age", "Sex"])
        .assign(
            correct=lambda d: d["correct"].astype(int),
            post=lambda d: d["post"].astype(int),
            study_arm=lambda d: d["study_arm"].astype(int).astype(str),
        )
    )


def summarize_observed(long_df: pd.DataFrame, outcome_label: str) -> pd.DataFrame:
    obs = (
        long_df.groupby(["study_arm", "post"], as_index=False)
        .agg(n_correct=("correct", "sum"), n=("correct", "count"))
    )
    rows = []
    for row in obs.itertuples(index=False):
        lo, hi = wilson_ci_pct(int(row.n_correct), int(row.n))
        rows.append(
            {
                "outcome": outcome_label,
                "arm": ARM_LABELS[row.study_arm],
                "phase": "Pre" if row.post == 0 else "Post",
                "pct": row.n_correct / row.n * 100,
                "ci_lo": lo,
                "ci_hi": hi,
            }
        )
    return pd.DataFrame(rows)


def glmer_interaction_p(long_df: pd.DataFrame) -> dict[str, float]:
    r("suppressPackageStartupMessages(library(lme4))")
    with localconverter(default_converter + pandas2ri.converter):
        r.assign("long_df", long_df.copy())
    r("""
    long_df$study_arm <- relevel(factor(long_df$study_arm), ref = "4")
    long_df$vignette <- factor(long_df$vignette, levels = c("A", "B", "C", "D", "E"))
    long_df$prolific_id <- factor(long_df$prolific_id)
    glmer_model <- glmer(
      correct ~ post * study_arm + (1 | prolific_id) + (1 | vignette),
      data = long_df,
      family = binomial,
      control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
    )
    """)
    with localconverter(default_converter + pandas2ri.converter):
        coef_df = r("as.data.frame(coef(summary(glmer_model)))")
    inter = coef_df.loc[coef_df.index.str.startswith("post:study_arm"), "Pr(>|z|)"]
    return {
        ARM_LABELS[idx.replace("post:study_arm", "")]: float(p)
        for idx, p in inter.items()
    }


def build_figure3_estimates(df: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for spec in FIG3_OUTCOMES:
        long_df = make_long_outcome(df, spec["pre_cols"], spec["post_cols"])
        obs = summarize_observed(long_df, spec["outcome"])
        interaction_p = glmer_interaction_p(long_df)
        obs["interaction_p"] = obs["arm"].map(interaction_p)
        parts.append(obs)
    out = pd.concat(parts, ignore_index=True)
    out["arm"] = pd.Categorical(out["arm"], categories=ARM_ORDER, ordered=True)
    out["phase"] = pd.Categorical(out["phase"], categories=["Pre", "Post"], ordered=True)
    return out


def star_arms_for_panel(panel_df: pd.DataFrame) -> dict[str, str]:
    stars = {}
    for arm in ARM_ORDER:
        if arm == "Control":
            continue
        p = panel_df.loc[panel_df["arm"] == arm, "interaction_p"].iloc[0]
        label = p_to_stars(p)
        if label:
            stars[arm] = label
    return stars


def plot_arm_panel(ax, panel_df, title, *, show_ylabel, star_arms):
    phase_order = {"Pre": 0, "Post": 1}
    max_top = 0.0
    for arm in ARM_ORDER:
        sub = panel_df[panel_df["arm"] == arm].sort_values("phase", key=lambda s: s.map(phase_order))
        color = ARM_COLORS[arm]
        jitter = ARM_JITTER[arm]
        xs = sub["phase"].map(PHASE_X).to_numpy() + jitter
        ys = sub["pct"].to_numpy()
        ax.plot(xs, ys, color=color, linewidth=1.5, solid_capstyle="round", zorder=2)
        for row, x in zip(sub.itertuples(index=False), xs):
            yerr = np.array([[row.pct - row.ci_lo], [row.ci_hi - row.pct]])
            ax.errorbar(
                x, row.pct, yerr=yerr, fmt="o", markersize=6.5, markeredgewidth=1.4,
                markerfacecolor=color, markeredgecolor=color, color=CI_COLOR,
                elinewidth=1.0, capsize=3, capthick=1.0, zorder=3,
            )
            max_top = max(max_top, row.ci_hi)
        if arm in star_arms:
            post_row = sub[sub["phase"] == "Post"].iloc[0]
            x_post = PHASE_X["Post"] + jitter
            y_star = post_row.ci_hi + 2.5
            ax.text(x_post, y_star, star_arms[arm], ha="center", va="bottom", fontsize=9, color="#333333", zorder=4)
            max_top = max(max_top, y_star + 2)
    ax.set_title(title, loc="left", fontsize=11, fontweight="bold", pad=8)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Pre", "Post"])
    ax.set_xlim(-0.2, 1.2)
    ax.set_ylabel("Percent correct" if show_ylabel else "")
    ax.set_ylim(0, min(100, max_top + 6))
    ax.yaxis.grid(True, linestyle="-", alpha=0.25, color="#cccccc")
    ax.set_axisbelow(True)


df = pd.read_excel(DATA_PATH)
estimates_df = build_figure3_estimates(df)
print(estimates_df.sort_values(["outcome", "arm", "phase"]).round(4).to_string(index=False))

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 10, "axes.spines.top": False,
    "axes.spines.right": False, "axes.linewidth": 0.8,
})
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8), sharey=True)
fig.subplots_adjust(wspace=0.08, top=0.82, bottom=0.14, left=0.08, right=0.98)
for ax, spec in zip(axes, FIG3_OUTCOMES):
    panel = estimates_df[estimates_df["outcome"] == spec["outcome"]]
    plot_arm_panel(
        ax, panel, spec["outcome"],
        show_ylabel=(ax is axes[0]),
        star_arms=star_arms_for_panel(panel),
    )
fig.legend(
    handles=[
        mlines.Line2D([0], [0], marker="o", color="w", markerfacecolor=ARM_COLORS[a],
                      markeredgecolor=ARM_COLORS[a], markersize=6.5, label=a, linestyle="None")
        for a in ARM_ORDER
    ],
    loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.0),
)
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(PNG_PATH, dpi=300, bbox_inches="tight")
fig.savefig(PDF_PATH, bbox_inches="tight")
print(f"Saved {PNG_PATH}")
print(f"Saved {PDF_PATH}")
plt.show()
